Import the necessary Libraries.Read diabetic patients file. And see the header of the file.

In [12]:
import pandas as pd
import numpy as np


In [13]:

records=pd.read_csv(r"Diabetes_record_linear.csv")

In [14]:
print(records.head())

   BMI   BP  Cholesterol  LDL  Target
0   26   99          165   75     524
1   39   65          285   92     717
2   34   70          226   77     655
3   31  114          274   63     732
4   21   96          198   68     606


In [15]:
print(records.isnull().sum())

BMI            0
BP             0
Cholesterol    0
LDL            0
Target         0
dtype: int64


Select the features (X) and the actual output (y) from the dataset. Scale the Features because their is variation between the feature.

In [16]:
y=records["Target"]
x=records.drop(columns=["Target"])
train_mean=x.mean()
stand_dev=x.std()
train_normilization=(x-train_mean)/stand_dev
print(train_normilization.head())

        BMI        BP  Cholesterol       LDL
0 -0.405526  0.496449    -1.369036 -1.120368
1  1.587606 -1.438138     1.234932 -0.713657
2  0.821017 -1.153640    -0.045352 -1.072520
3  0.361063  1.349944     0.996235 -1.407458
4 -1.172115  0.325750    -0.652945 -1.287837


Compute cost function. Below there are two ways given one for better understanding but it was very slow so to speed this up we use the vectorized version of this

In [17]:

def cost_fun(x,y,w,b):
    summation=0
    m=x.shape[0]
    w_x=np.dot(x,w)+b
    cost=(w_x-y)**2
    summation=np.sum(cost)/(2*m)
    return summation
# Cost Function
# def cost_fun(x,y,w,b):
#     summation=0
#     m=x.shape[0]
#     for i in range(m):
#         w_x=np.dot(x[i],w)+b
#         cost=(w_x-y[i])**2
#         summation+=cost/(2*m)
#     return summation






Calculate derivation: the given block have two types of codes. if we use the commented part we use the vectorized version because -> Instead of running millions slow lines of Python loop code, NumPy sends the entire matrix block to highly optimized C-code background libraries. The computer calculates the entire gradient step in a single operation, allowing your 100,000 iterations to finish in under a second.

In [18]:
def derivative(x, y, w, b):
    m, n = x.shape
    f_wb = np.dot(x, w) + b
    err = f_wb - y
    dj_dw = np.dot(x.T, err) / m
    dj_db = np.sum(err) / m
    
    return dj_dw, dj_db

# def derivative(x,y,w,b):
#     m,n=x.shape
#     cost_w=np.zeros((n,))
#     cost_b=0
#     for i in range(m):
#         w_x=(np.dot(x[i],w)+b)-y[i]
#         for j in range(n):
#             cost_w[j]=cost_w[j]+w_x*x[i,j]
#         cost_b+=(w_x)
#     cost_w=cost_w/m
#     cost_b=cost_b/m
#     return cost_w,cost_b


Compute gradient descent

In [19]:
def gradient_descent(x,y,w,b,num_iter,alpha,derivative_function,cost_function):
    for i in range(num_iter+1):
        gradient_w,gradient_b=derivative_function(x,y,w,b)
        compute_cost=cost_function(x,y,w,b)
        w=w-alpha*gradient_w
        b=b-alpha*gradient_b
        if i % 1000 == 0:
            print(f"Iteration {i}: Cost={compute_cost:.4f}")
    print("-------------------------\n --------------------")  
    return w,b


Prediction and compare the predicited and actual values

In [20]:
x = train_normilization.values
y = y.values.reshape(-1)

num_iter = 10000
alpha = 0.001

weight = np.array([0, 0, 0, 0])
bias = 0

print("Linear Regression in progress...........")

w, b = gradient_descent(
    x, y, weight, bias,
    num_iter, alpha,
    derivative, cost_fun
)

m = x.shape[0]

for i in range(m):
    prediction = np.dot(w, x[i]) + b
    print(f"prediction: {prediction:.2f} target: {y[i]}")
# Predictions
predictions = np.dot(x, w) + b



Linear Regression in progress...........
Iteration 0: Cost=249929.1250
Iteration 1000: Cost=34138.5014
Iteration 2000: Cost=4983.0987
Iteration 3000: Cost=1042.2579
Iteration 4000: Cost=509.3476
Iteration 5000: Cost=437.2466
Iteration 6000: Cost=427.4855
Iteration 7000: Cost=426.1630
Iteration 8000: Cost=425.9837
Iteration 9000: Cost=425.9593
Iteration 10000: Cost=425.9560
-------------------------
 --------------------
prediction: 546.26 target: 524
prediction: 712.53 target: 717
prediction: 608.00 target: 655
prediction: 689.17 target: 732
prediction: 555.07 target: 606
prediction: 701.43 target: 685
prediction: 545.07 target: 513
prediction: 616.23 target: 609
prediction: 743.60 target: 739
prediction: 668.65 target: 694
prediction: 521.55 target: 473
prediction: 906.40 target: 953
prediction: 848.41 target: 845
prediction: 565.86 target: 549
prediction: 820.64 target: 798
prediction: 761.88 target: 716
prediction: 525.36 target: 545
prediction: 635.85 target: 637
prediction: 822.43

Lets computes the R^2 Score. The code determines how closely your machine learning model's predictions align with your actual data points.

In [21]:
# R²
ss_res = np.sum((y - predictions) ** 2)
ss_tot = np.sum((y - np.mean(y)) ** 2)

r2 = 1 - (ss_res / ss_tot)

print("R² Score:", r2)
print("R² Percentage:", r2 * 100, "%")

R² Score: 0.9167744029969908
R² Percentage: 91.67744029969907 %


User INPUT (user enter the data and model predicts the targets value)

In [22]:
BMI = float(input("Enter BMI: "))
BP = float(input("Enter BP: "))
Cholesterol = float(input("Enter Cholesterol: "))
LDL = float(input("Enter LDL: "))

new_patient = pd.DataFrame({
    "BMI": [BMI],
    "BP": [BP],
    "Cholesterol": [Cholesterol],
    "LDL": [LDL]
})

# Normalize using training mean and standard deviation
new_patient_normalized = (
    new_patient - train_mean
) / stand_dev

# Convert to NumPy
new_patient_record = new_patient_normalized.values.flatten()

# Prediction
prediction = np.dot(w, new_patient_record) + b

print(f"\nPredicted Target: {prediction:.2f}")


Predicted Target: 546.26
